# FramePack pipeline

1. Run the cell below. The first run in a session installs everything and downloads ~75 GB of models in parallel.
2. Open the public link. Upload your start frames (several images or a ZIP), write an **end-frame prompt**, and press **Run**.
3. Check the end frames in the review panel, re-roll any bad ones, then press **Approve → make videos**.

Everything lands in `My Drive/<Drive_folder>/<batch>/`. If the runtime disconnects, re-run the cell and pick the batch under **Resume a batch**.

In [ ]:
#@title <font size="6" color="orange">FramePack pipeline</font>
#@markdown #### Start frames → AI end frames (Qwen-Image-Edit) → FramePack videos. Click run, then open the **public** URL (`https://xxxxxxx.gradio.live`).
Drive_folder = 'pongers' #@param {type: "string"}
#@markdown Folder in My Drive where each batch (start/end frames, videos, manifest) is saved. Unfinished batches can be resumed from the app.
Prefetch_F1 = False #@param {type: "boolean"}
#@markdown Also download the FramePack-F1 weights (~26 GB) now, to compare them with the default end-frame-trained model.
High_VRAM = False #@param {type: "boolean"}
#@markdown Keep every model on the GPU when it has 80 GB (A100 80GB / H100). Leave off on 40 GB and smaller cards.
NGROK = '' #@param {type: "string"}
#@markdown Use NGROK if Gradio has connection issues. Get API key [here](https://dashboard.ngrok.com/get-started/your-authtoken). You can also use the notebook secret `NGROK`.
Branch = 'pipeline' #@param {type: "string"}

import os
import threading

REPO = 'https://github.com/kdeary/FramePack'
FP_DIR = '/content/FramePack'
COMFY_DIR = '/content/ComfyUI'


def ngrok_thread(port):
  %pip install pyngrok
  import time
  import socket

  # Wait for the app to open port
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()

  # Write config
  NGROK_CONFIG=f"""version: 3
endpoints:
    - name: gradio
      upstream:
        url: http://localhost:{port}
      traffic_policy:
        inbound:
            - name: "Remove X-Forwarded-Host"
              actions:
                - type: remove-headers
                  config:
                    headers:
                        - "x-forwarded-host"
agent:
    authtoken: {NGROK}
  """
  with open("/content/ngrok_config.yaml", "w") as f:
    f.write(NGROK_CONFIG)

  !ngrok start gradio --config /content/ngrok_config.yaml --log=stdout


# Get secrets
from google.colab import userdata
for secret in ('NGROK', 'HF_TOKEN'):
  try:
    value = userdata.get(secret)
  except Exception:
    continue
  if secret == 'NGROK':
    NGROK = NGROK or value
  else:
    os.environ['HF_TOKEN'] = value  # faster, rate-limit-free model downloads
  print(f'Used {secret} secret.')

from google.colab import drive
drive.mount('/content/drive')

# Code
if os.path.exists(FP_DIR):
  !git -C {FP_DIR} pull -q
else:
  !git clone -q -b {Branch} {REPO} {FP_DIR}

# Environments + models (skipped if this session already has them)
os.environ['PREFETCH_F1'] = '1' if Prefetch_F1 else '0'
os.environ['FP_DIR'] = FP_DIR
os.environ['COMFY_DIR'] = COMFY_DIR
!bash {FP_DIR}/colab/setup.sh
if not os.path.exists('/content/.pipeline_setup_done'):
  raise SystemExit('Setup failed, see the messages above.')

# Outputs go straight to Drive
output_path = f'/content/drive/MyDrive/{Drive_folder}'
os.makedirs(output_path, exist_ok=True)
!rm -rf {FP_DIR}/outputs
!ln -s {output_path} {FP_DIR}/outputs

# Run
cmd = (f'cd {FP_DIR} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True /content/venv-fp/bin/python demo_gradio_f1.py'
       f' --comfy_dir {COMFY_DIR} --comfy_python /content/venv-comfy/bin/python')
if High_VRAM:
  cmd += ' --high_vram auto'
if NGROK:
  port = 7860
  threading.Thread(target=ngrok_thread, daemon=True, args=(port,)).start()
else:
  cmd += ' --share'

print(cmd)
!{cmd}
